In [0]:
%run "../00.common/01.environment-config.py"

In [0]:
# Databricks notebook source

# MAGIC %md
# MAGIC # Build Constructors Dimension
# MAGIC
# MAGIC 1. Read Silver `constructors` data
# MAGIC 2. Read Gold `ref_nationality_region` data
# MAGIC 3. Join using `nationality`
# MAGIC 4. Select the required business columns
# MAGIC 5. Write the transformed data to Gold `dim_constructors`

# COMMAND ----------

# MAGIC %run "../00.common/01.environment-config.py"

# COMMAND ----------

from pyspark.sql import functions as F

# COMMAND ----------

# ============================================================
# SOURCE AND TARGET PATHS
# ============================================================

silver_path = f"{s3_root_path}/Silver/Nisarga/constructors"

ref_path = f"{s3_root_path}/Gold/Nisarga/ref_nationality_region"

gold_path = f"{s3_root_path}/Gold/Nisarga/dim_constructors"

print("Silver Path    :", silver_path)
print("Reference Path :", ref_path)
print("Gold Path      :", gold_path)

# COMMAND ----------

# ============================================================
# STEP 1 - READ SOURCE DATA
# ============================================================

constructors_df = (
    spark.read
    .format("delta")
    .load(silver_path)
)

ref_nationality_region_df = (
    spark.read
    .format("delta")
    .load(ref_path)
)

display(constructors_df)
display(ref_nationality_region_df)

# COMMAND ----------

# ============================================================
# STEP 2 - JOIN CONSTRUCTORS WITH NATIONALITY REGION
# ============================================================

dim_constructors_df = (
    constructors_df
    .join(
        ref_nationality_region_df,
        constructors_df.nationality == ref_nationality_region_df.nationality,
        "left"
    )
    .select(
        constructors_df.constructor_id,
        constructors_df.name,
        constructors_df.nationality,
        ref_nationality_region_df.region
    )
)

display(dim_constructors_df)

# COMMAND ----------

# ============================================================
# STEP 3 - REMOVE DUPLICATES
# ============================================================

dim_constructors_df = (
    dim_constructors_df
    .dropDuplicates(["constructor_id"])
)

display(dim_constructors_df)

# COMMAND ----------

# ============================================================
# STEP 4 - WRITE TO GOLD
# ============================================================

(
    dim_constructors_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(gold_path)
)

print("Gold Constructors data successfully written to:")
print(gold_path)

# COMMAND ----------

# ============================================================
# STEP 5 - VALIDATE GOLD DATA
# ============================================================

gold_constructors_df = (
    spark.read
    .format("delta")
    .load(gold_path)
)

display(gold_constructors_df)

print("Silver Constructors Records :", constructors_df.count())
print("Gold Constructors Records   :", gold_constructors_df.count())

# COMMAND ----------

# ============================================================
# STEP 6 - CHECK NATIONALITY REGION MAPPING
# ============================================================

display(
    gold_constructors_df
    .groupBy("region")
    .count()
    .orderBy("region")
)